In [1]:
import os
import pickle
import tarfile
import datetime
import numpy as np
import urllib.request
import sklearn.metrics
import tensorflow as tf
import matplotlib.pyplot as plt
import zipfile

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
              tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [3]:
%load_ext tensorboard

In [4]:
BATCH_SIZE = 128
HISTORY_DIR = './history'
os.makedirs(HISTORY_DIR, exist_ok=True)

In [5]:
def download_data():
    if not os.path.exists('tiny-imagenet-200'):
        urllib.request.urlretrieve('http://cs231n.stanford.edu/tiny-imagenet-200.zip', 'tiny-imagenet-200.zip')
        file = zipfile.ZipFile('tiny-imagenet-200.zip', 'r')
        file.extractall()

In [6]:
from tensorflow.keras.preprocessing import image_dataset_from_directory
IMG_SIZE = 64
NUM_CLASSES = 200

In [7]:
# уже загружено
# download_data()

In [8]:
# Сделано
# # настроим правильную структуру папок для val
# import os
# import shutil

# val_dir = 'tiny-imagenet-200/val'
# val_images_dir = os.path.join(val_dir, 'images')
# val_annotations_file = os.path.join(val_dir, 'val_annotations.txt')

# # Чтение аннотаций
# with open(val_annotations_file, 'r') as f:
#     for line in f:
#         parts = line.strip().split('\t')
#         img_file = parts[0]
#         class_name = parts[1]
        
#         class_dir = os.path.join(val_dir, class_name)
#         os.makedirs(class_dir, exist_ok=True)
        
#         src_path = os.path.join(val_images_dir, img_file)
#         dst_path = os.path.join(class_dir, img_file)
#         shutil.move(src_path, dst_path)

# # Удалим папку images — она теперь не нужна
# shutil.rmtree(val_images_dir)

In [9]:
train_dataset = image_dataset_from_directory(
    'tiny-imagenet-200/train',
    label_mode='categorical',       # для one-hot меток
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

Found 100001 files belonging to 200 classes.


In [10]:
val_dataset = image_dataset_from_directory(
    'tiny-imagenet-200/val',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 10000 files belonging to 200 classes.


In [11]:
from tensorflow.keras import layers, models, optimizers, regularizers

# Аугментация
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

In [18]:
# аугментация + нормализация
def preprocess(images, labels):
    images = tf.cast(images, tf.float32) / 255.0  # нормализация [0, 1]
    return images, labels

In [13]:
train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))
train_dataset = train_dataset.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.map(preprocess).prefetch(tf.data.AUTOTUNE)

In [19]:
L2 = regularizers.l2(1e-4)
# layers.Conv2D(64, (3, 3), padding='same', activation='relu', kernel_regularizer=L2),

In [27]:
model = models.Sequential([
    layers.InputLayer(input_shape=(64, 64, 3)),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.4),

    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.5),

    layers.Flatten(),
    layers.Dense(512, activation='relu', kernel_regularizer=L2),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

In [28]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_24 (Conv2D)                   │ (None, 64, 64, 64)          │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_24               │ (None, 64, 64, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_25 (Conv2D)                   │ (None, 64, 64, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_25               │ (None, 64, 64, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_26 (Conv2D)                   │ (None, 32, 32, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_26               │ (None, 32, 32, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_27 (Conv2D)                   │ (None, 32, 32, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_27               │ (None, 32, 32, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_28 (Conv2D)                   │ (None, 16, 16, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_28               │ (None, 16, 16, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_29 (Conv2D)                   │ (None, 16, 16, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_29               │ (None, 16, 16, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 8, 8, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_18 (Dropout)                 │ (None, 8, 8, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 9,640,712 (36.78 MB)

 Trainable params: 9,638,920 (36.77 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [29]:
# model.compile(loss=..., metrics=[...], optimizer=...)

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

In [30]:
logdir = os.path.join(HISTORY_DIR, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

In [24]:
%tensorboard --logdir $logdir

In [25]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(logdir, 'model_tiny.keras'),
    save_best_only=True
)

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    os.path.join(logdir, 'logs'),    
)

early_stop_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=15,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1,
    min_lr=1e-6
)

In [31]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=[model_checkpoint_callback, 
               tensorboard_callback, 
               early_stop_callback,
               reduce_lr]
)

Epoch 1/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1082s 1s/step - accuracy: 0.0051 - loss: 5.5926 - val_accuracy: 0.0054 - val_loss: 5.3699 - learning_rate: 0.0010
Epoch 2/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1070s 1s/step - accuracy: 0.0044 - loss: 5.3588 - val_accuracy: 0.0053 - val_loss: 5.3322 - learning_rate: 0.0010
Epoch 3/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1068s 1s/step - accuracy: 0.0044 - loss: 5.3665 - val_accuracy: 0.0058 - val_loss: 5.3614 - learning_rate: 0.0010
Epoch 4/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1070s 1s/step - accuracy: 0.0041 - loss: 5.3547 - val_accuracy: 0.0050 - val_loss: 5.3241 - learning_rate: 0.0010
Epoch 5/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1069s 1s/step - accuracy: 0.0044 - loss: 5.3218 - val_accuracy: 0.0061 - val_loss: 5.5943 - learning_rate: 0.0010
Epoch 6/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1069s 1s/step - accuracy: 0.0042 - loss: 5.4010 - val_accuracy: 0.0050 - val_loss: 5.3399 - learning_rate: 0.0010
Epoch 7/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 1069s 1s/step - accuracy: 0.0

KeyboardInterrupt: 

In [ ]:
y_true = []
y_true = []
for _, labels in val_dataset.unbatch():
    y_true.append(np.argmax(labels.numpy()))
y_true = np.array(y_true)


y_pred_probs = model.predict(val_dataset)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
_, ax = plt.subplots(figsize=(75, 75))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)
plt.tight_layout()
plt.savefig('valid.png')